# Using Joblib Models

This notebook demonstrates how to load and use the three saved joblib models:
1. **preprocessing_pipeline** — transforms raw features (PowerTransformer + PCA)
2. **kmeans_model** — assigns companies to clusters (K=8)
3. **cluster_id_classifier** — predicts cluster membership via Logistic Regression

These models can be used independently or in sequence on new data.

## Setup

In [19]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from pathlib import Path

# Set paths
DATA_DIR = Path('../Data')
TRAIN_DATA = Path('../Data/train_data.csv')

# Load the training data for demonstration
df = pd.read_csv(TRAIN_DATA)
df.columns = df.columns.str.strip()  # clean whitespace

print(f'Loaded training data: {df.shape}')
print(f'Columns: {df.shape[1]}')
print(f'\nFirst few rows:')
df.head()

Loaded training data: (5807, 97)
Columns: 97

First few rows:


,Index,Bankrupt?,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
0,0,0,0.450397,0.504034,0.506986,0.594640,0.594640,0.998906,0.797293,0.809239,...,0.780554,0.004919,0.623634,0.594641,0.838869,0.279036,0.026788,0.565144,1,0.032464
1,1,0,0.530005,0.572885,0.574763,0.605695,0.605558,0.999058,0.797512,0.809399,...,0.819963,0.005968,0.624171,0.605690,0.841869,0.279040,0.026801,0.565205,1,0.032442
2,2,0,0.571150,0.620148,0.624177,0.612275,0.612282,0.999163,0.797654,0.809533,...,0.839128,0.006022,0.625306,0.612271,0.843294,0.278927,0.026816,0.565276,1,0.033034
3,3,0,0.483401,0.556694,0.536164,0.602445,0.602445,0.999035,0.797458,0.809380,...,0.806477,0.002177,0.621610,0.602444,0.841891,0.293391,0.027063,0.566190,1,0.015406
4,4,0,0.510359,0.537287,0.552546,0.600023,0.600023,0.999009,0.797406,0.809313,...,0.799277,0.001124,0.623993,0.600019,0.840313,0.279878,0.026880,0.565549,1,0.028858


## Load the Three Joblib Models

In [20]:
from sklearn.base import BaseEstimator, TransformerMixin

class ColumnDropper(BaseEstimator, TransformerMixin):
    """Drops specified columns. Used as the first step of the pipeline so test data
    goes through the same column-drop step as training data."""
    def __init__(self, columns_to_drop):
        self.columns_to_drop = columns_to_drop

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X.drop(columns=[c for c in self.columns_to_drop if c in X.columns])
        return X

In [21]:
# Load all three models
preprocessing_pipeline = joblib.load(DATA_DIR / 'preprocessing_pipeline.joblib')
kmeans_model = joblib.load(DATA_DIR / 'kmeans_model.joblib')
cluster_id_classifier = joblib.load(DATA_DIR / 'cluster_id_classifier.joblib')

print('[OK] All three models loaded successfully')
print(f'\nPreprocessing pipeline steps: {preprocessing_pipeline.named_steps.keys()}')
print(f'Preprocessing output shape: 47 PCA components')
print(f'\nKMeans clusters: {kmeans_model.n_clusters}')
print(f'\nCluster ID classifier: {cluster_id_classifier.named_steps.keys()}')

[OK] All three models loaded successfully

Preprocessing pipeline steps: dict_keys(['drop_cols', 'power_transform', 'pca'])
Preprocessing output shape: 47 PCA components

KMeans clusters: 8

Cluster ID classifier: dict_keys(['scaler', 'clf'])


## Extract Features from Raw Data

In [22]:
# Extract features and target from training data
y_true = df['Bankrupt?'].copy()
X_raw = df.drop(columns=['Index', 'Bankrupt?'])

print(f'Raw features shape: {X_raw.shape}')
print(f'Target shape: {y_true.shape}')
print(f'\nTarget distribution:')
print(y_true.value_counts())

Raw features shape: (5807, 95)
Target shape: (5807,)

Target distribution:
Bankrupt?
0    5609
1     198
Name: count, dtype: int64


## Using the Preprocessing Pipeline

In [23]:
# The preprocessing pipeline handles:
# 1. Dropping zero-variance and highly-correlated features
# 2. Yeo-Johnson PowerTransformer + StandardScaler
# 3. PCA (95% variance, 47 components)

X_preprocessed = preprocessing_pipeline.transform(X_raw)

print(f'Input shape:  {X_raw.shape}')
print(f'Output shape: {X_preprocessed.shape}')
print(f'\nPreprocessed data (first 5 rows, first 10 components):')
print(pd.DataFrame(X_preprocessed[:5, :10]).round(3))

Input shape:  (5807, 95)
Output shape: (5807, 47)

Preprocessed data (first 5 rows, first 10 components):
       0      1      2      3      4      5      6      7      8      9
0  2.700 -0.896 -0.803 -0.603  0.201 -1.551  2.438 -0.057  0.832  0.064
1 -0.223  1.489 -0.580  0.047 -0.463 -0.042 -0.902  0.993 -0.079  1.006
2 -2.141  0.200 -1.397  0.274  0.629  0.110 -0.842 -0.342  0.591  1.313
3  4.642  3.466  2.807  1.272 -3.475  3.052 -0.139 -3.157  0.885 -0.898
4  0.262  4.231  0.972 -0.637 -0.461 -1.692  1.836  0.804 -0.084  0.198


## Using K-mean clustering

In [24]:
# Use KMeans to predict cluster for each company
# Note: KMeans expects the preprocessed data (47 PCA components)

cluster_assignments = kmeans_model.predict(X_preprocessed)

print(f'Cluster assignments shape: {cluster_assignments.shape}')
print(f'Unique clusters: {np.unique(cluster_assignments)}')
print(f'\nCluster distribution:')
print(pd.Series(cluster_assignments).value_counts().sort_index())

Cluster assignments shape: (5807,)
Unique clusters: [0 1 2 3 4 5 6 7]

Cluster distribution:
0     610
1    1749
2     904
3    1024
4      33
5     916
6       7
7     564
Name: count, dtype: int64


# Using the Cluster ID Classifier (Routing)

In [25]:
# The cluster ID classifier predicts which cluster using raw (pre-PCA) features
# This allows interpretation via financial ratios instead of PCA components

# The classifier was trained WITHOUT zero-variance features (Net Income Flag was dropped)
# So we need to drop it before prediction
X_for_classifier = X_raw.drop(columns=['Net Income Flag'], errors='ignore').copy()

cluster_predictions = cluster_id_classifier.predict(X_for_classifier)
cluster_probs = cluster_id_classifier.predict_proba(X_for_classifier)

print(f'Cluster predictions shape: {cluster_predictions.shape}')
print(f'\nCluster distribution (from classifier):')
print(pd.Series(cluster_predictions).value_counts().sort_index())
print(f'\nPrediction confidence (max probability per sample):')
print(f'  Mean: {cluster_probs.max(axis=1).mean():.4f}')
print(f'  Min:  {cluster_probs.max(axis=1).min():.4f}')
print(f'  Max:  {cluster_probs.max(axis=1).max():.4f}')

Cluster predictions shape: (5807,)

Cluster distribution (from classifier):
0     608
1    1778
2     895
3    1021
4      33
5     906
6       7
7     559
Name: count, dtype: int64

Prediction confidence (max probability per sample):
  Mean: 0.9139
  Min:  0.3483
  Max:  1.0000


## Using Models on a New Sample

In [26]:
# Demonstrate the recommended pipeline for new samples:
# Step 1: Preprocess
# Step 2: Assign cluster with classifier

# Take first 10 companies as example
X_new = X_raw.iloc[:10].copy()

# Step 1: Preprocess raw features
# Drop zero-variance feature first
X_new_for_classifier = X_new.drop(columns=['Net Income Flag'], errors='ignore')

# Step 2: Assign cluster using classifier
# (Classifier uses raw features directly - simpler and more interpretable)
clusters_classifier = cluster_id_classifier.predict(X_new_for_classifier)
clusters_classifier_probs = cluster_id_classifier.predict_proba(X_new_for_classifier)

# Create results
new_results = pd.DataFrame({
    'Company_Index': df['Index'].iloc[:10].values,
    'True_Bankrupt': y_true.iloc[:10].values,
    'Predicted_Cluster': clusters_classifier,
    'Confidence': clusters_classifier_probs.max(axis=1)
})

print('Predictions for first 10 companies:')
print(new_results.to_string(index=False))

Predictions for first 10 companies:
 Company_Index  True_Bankrupt  Predicted_Cluster  Confidence
             0              0                  5    0.820250
             1              0                  1    0.882605
             2              0                  1    0.994185
             3              0                  2    0.990437
             4              0                  3    0.999989
             5              0                  3    0.999852
             6              0                  2    0.991544
             7              0                  5    0.999931
             8              0                  1    0.988275
             9              0                  7    0.981385


## Summary of libs

**The three models work independently:**

1. **preprocessing_pipeline.joblib**
   - Input: Raw features (95 columns from train_data.csv)
   - Output: 47 PCA components
   - Used by: KMeans clustering

2. **kmeans_model.joblib**
   - Input: 47 PCA components (from preprocessing pipeline)
   - Output: Cluster assignment (0-7)
   - Use when: You want geometric clustering in PCA space

3. **cluster_id_classifier.joblib**
   - Input: Raw features (95 columns)
   - Output: Predicted cluster (0-7) + probabilities
   - Use when: You want interpretable feature importance or confidence scores